In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/simuteh-2330511039/src/

Mounted at /content/drive
/content/drive/MyDrive/simuteh-2330511039/src


In [ ]:
# Kode 4.1 Verifikasi konvolusi manual

In [3]:
import time
import cv2
import numpy as np

# Catatan: Jika fungsi 'baca' didefinisikan di file lain, pastikan untuk mengimpornya.
# Sebagai fallback, jika 'baca' adalah pembaca gambar biasa, kita bisa mendefinisikannya:
def baca(path):
    return cv2.imread(path)

def konvolusi(img, k):            # fungsi dari slide P4
    kh, kw = k.shape; ph, pw = kh // 2, kw // 2
    pad = np.pad(img.astype(float), ((ph, ph), (pw, pw)), mode="edge")
    out = np.zeros(img.shape)
    for y in range(img.shape[0]):
        for x in range(img.shape[1]):
            out[y, x] = np.sum(pad[y:y+kh, x:x+kw] * k)
    return np.clip(np.round(out), 0, 255).astype(np.uint8)

def konvolusi_sejati(img, k):     # kernel dibalik 180 derajat dahulu
    return konvolusi(img, np.flip(k))

nama = "teh_041.png"
bersih = cv2.cvtColor(baca(f"../data/sintetis/bersih/{nama}"),
                      cv2.COLOR_BGR2GRAY)
md = cv2.imread(f"../data/sintetis/mask_daun/{nama}", 0)
mb = cv2.imread(f"../data/sintetis/mask_bercak/{nama}", 0)

for nm, k in {"rerata": np.ones((3, 3)) / 9,
              "asimetris": np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]], float)
              }.items():
    t0 = time.perf_counter(); a = konvolusi(bersih, k)
    t_manual = time.perf_counter() - t0
    t0 = time.perf_counter()
    b = cv2.filter2D(bersih, -1, k, borderType=cv2.BORDER_REPLICATE)
    t_cv = time.perf_counter() - t0
    beda = np.abs(a.astype(int) - b.astype(int)).max()
    c = konvolusi_sejati(bersih, k)
    beda2 = np.abs(c.astype(int) - b.astype(int)).max()
    print(f"{nm}: selisih manual vs filter2D={beda}, "
          f"sejati vs filter2D={beda2}, "
          f"waktu manual={t_manual:.3f} s, filter2D={t_cv:.5f} s")

rerata: selisih manual vs filter2D=0, sejati vs filter2D=0, waktu manual=2.137 s, filter2D=0.00856 s
asimetris: selisih manual vs filter2D=0, sejati vs filter2D=255, waktu manual=3.762 s, filter2D=0.00056 s


In [ ]:
# Kode 4.2 Tabel penapis x derau x PSNR x waktu x KB

In [8]:
import pandas as pd
rng = np.random.default_rng(42)

def derau_gauss(img, sd):
    return np.clip(img + rng.normal(0, sd, img.shape), 0, 255).astype(np.uint8)

def derau_sp(img, p):                  # p = proporsi piksel rusak
    out = img.copy(); m = rng.random(img.shape)
    out[m < p / 2] = 0; out[m > 1 - p / 2] = 255
    return out

def ukur_waktu(f, x, ulang=20):        # rerata beberapa kali, dalam ms
    t = []
    for _ in range(ulang):
        t0 = time.perf_counter(); f(x); t.append(time.perf_counter() - t0)
    return 1000 * np.mean(t)

def kontras_bercak(img, mask_daun, mask_bercak):
    # Menghitung rata-rata intensitas piksel di area daun dan area bercak
    daun_mean = img[mask_daun == 255].mean() if np.any(mask_daun == 255) else 0.0
    bercak_mean = img[mask_bercak == 255].mean() if np.any(mask_bercak == 255) else 0.0
    return np.abs(daun_mean - bercak_mean)

derau = {"gauss_10": derau_gauss(bersih, 10), "gauss_20": derau_gauss(bersih, 20),
         "gauss_40": derau_gauss(bersih, 40), "sp_4%": derau_sp(bersih, 0.04),
         "sp_10%": derau_sp(bersih, 0.10)}
penapis = {"rerata_3": lambda x: cv2.blur(x, (3, 3)),
           "rerata_5": lambda x: cv2.blur(x, (5, 5)),
           "gauss_5": lambda x: cv2.GaussianBlur(x, (5, 5), 1.0),
           "median_3": lambda x: cv2.medianBlur(x, 3),
           "median_5": lambda x: cv2.medianBlur(x, 5),
           "median_9": lambda x: cv2.medianBlur(x, 9)}

baris = []
for dn, d in derau.items():
    baris.append([dn, "tanpa", cv2.PSNR(bersih, d), 0.0,
                  kontras_bercak(d, md, mb)])
    for pn, f in penapis.items():
        out = f(d)
        baris.append([dn, pn, cv2.PSNR(bersih, out), ukur_waktu(f, d),
                      kontras_bercak(out, md, mb)])
tabel = pd.DataFrame(baris, columns=["derau", "penapis", "PSNR_dB",
                                     "waktu_ms", "KB"]).round(2)
print(tabel)
tabel.to_csv("../hasil/b4_tabel_penapis.csv", index=False)

       derau   penapis  PSNR_dB  waktu_ms     KB
0   gauss_10     tanpa    28.11      0.00  51.11
1   gauss_10  rerata_3    36.95      0.13  51.08
2   gauss_10  rerata_5    38.02      0.17  50.50
3   gauss_10   gauss_5    37.82      0.29  50.95
4   gauss_10  median_3    35.52      0.09  51.00
5   gauss_10  median_5    38.67      0.77  50.34
6   gauss_10  median_9    39.58     14.67  47.41
7   gauss_20     tanpa    22.10      0.00  51.94
8   gauss_20  rerata_3    31.44      0.13  51.94
9   gauss_20  rerata_5    34.66      0.16  51.34
10  gauss_20   gauss_5    32.60      0.28  51.82
11  gauss_20  median_3    29.74      0.09  51.88
12  gauss_20  median_5    33.56      0.73  51.33
13  gauss_20  median_9    36.02     14.58  48.16
14  gauss_40     tanpa    16.36      0.00  51.62
15  gauss_40  rerata_3    25.82      0.13  51.28
16  gauss_40  rerata_5    29.82      0.16  50.63
17  gauss_40   gauss_5    27.05      0.29  51.17
18  gauss_40  median_3    23.83      0.09  51.43
19  gauss_40  median

In [ ]:
# Grafik PSNR dan waktu komputasi

In [ ]:
# Kode 4.4 Pengaruh urutan penajaman

In [13]:
def unsharp(x, k=1.0):
 blur = cv2.GaussianBlur(x, (5, 5), 1.0)
 return cv2.addWeighted(x, 1 + k, blur, -k, 0)
noisy = derau["gauss_20"]
latar = md == 0 # area latar yang seharusnya rata
for nm, redam in {"Gaussian": lambda x: cv2.GaussianBlur(x, (5, 5), 1.0),
 "Median": lambda x: cv2.medianBlur(x, 5)}.items():
 A = redam(unsharp(noisy)) # tajamkan dulu, lalu redam
 B = unsharp(redam(noisy)) # redam dulu, lalu tajamkan
 print(f"{nm}: tajam->redam PSNR={cv2.PSNR(bersih, A):.2f} "
 f"sigma_latar={A[latar].std():.1f} | redam->tajam "
 f"PSNR={cv2.PSNR(bersih, B):.2f} sigma_latar={B[latar].std():.1f}")

Gaussian: tajam->redam PSNR=30.35 sigma_latar=7.9 | redam->tajam PSNR=30.20 sigma_latar=8.1
Median: tajam->redam PSNR=30.21 sigma_latar=8.0 | redam->tajam PSNR=31.17 sigma_latar=7.1


In [ ]:
# Langkah 4.6 Penapis pilihan pada 12 citra normal

In [14]:
import cv2, numpy as np, pandas as pd
from utils_cv import baca

# Penapis pilihan: Median 3x3
# Alasan: Efektif meredam derau sensor tanpa mengaburkan detail bercak kecil & komputasi sangat cepat.

# 12 citra kondisi normal (kelipatan 5 nomor: 001, 006, 011, ..., 056)
indeks_normal = [f"teh_{i:03d}.png" for i in range(1, 61, 5)]

baris = []
for nama in indeks_normal:
    citra_berderau = cv2.cvtColor(baca(f"../data/sintetis/citra/{nama}"), cv2.COLOR_BGR2GRAY)
    citra_bersih = cv2.cvtColor(baca(f"../data/sintetis/bersih/{nama}"), cv2.COLOR_BGR2GRAY)

    # Penerapan penapis Median 3x3
    citra_hasil = cv2.medianBlur(citra_berderau, 3)

    psnr_sebelum = cv2.PSNR(citra_bersih, citra_berderau)
    psnr_sesudah = cv2.PSNR(citra_bersih, citra_hasil)
    selisih = psnr_sesudah - psnr_sebelum

    baris.append([nama.replace(".png", ""), round(psnr_sebelum, 2), round(psnr_sesudah, 2), round(selisih, 2)])

# Tampilkan Tabel 16
tabel_16 = pd.DataFrame(baris, columns=["Citra", "PSNR sebelum (dB)", "PSNR sesudah (dB)", "Selisih (dB)"])
print(tabel_16.to_string(index=False))

  Citra  PSNR sebelum (dB)  PSNR sesudah (dB)  Selisih (dB)
teh_001              30.86              38.11          7.26
teh_006              33.00              40.04          7.04
teh_011              33.92              40.86          6.94
teh_016              30.53              37.82          7.28
teh_021              32.82              39.89          7.07
teh_026              30.70              37.97          7.27
teh_031              32.63              39.78          7.15
teh_036              31.02              38.27          7.25
teh_041              33.37              40.37          7.00
teh_046              33.03              39.86          6.83
teh_051              29.56              36.87          7.31
teh_056              35.61              42.10          6.49
